# Silver Layer : Deduplicate Sensor Readings

**Purpose:** Remove duplicate sensor readings and keep latest records

**Input:** `dev.bronze.sensor_readings_raw`

**Output:** `dev.silver.sensor_readings_deduplicated`

**Why dedup?**
- Network restries can cause duplicate events
- ~2% of our data is duplicates
- Duplicates skew analytics and ML features

## Configuration

In [0]:
# Configuration

CATALOG = "dev"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

INPUT_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.sensor_readings_raw"
OUTPUT_TABLE = f"{CATALOG}.{SCHEMA_SILVER}.sensor_readings_deduplicated"

print(f"Input: {INPUT_TABLE}")
print(f"Output: {OUTPUT_TABLE}")

## Load Bronze Data

In [0]:
# Read bronze table
bronze_df = spark.read.table(INPUT_TABLE)

print(f"Total records from Bronze: {bronze_df.count()}")
print("\nSchema:")
bronze_df.printSchema()

#Show sample
print("\nSample Data")
bronze_df.select("equipment_id", "sensor_type", "timestamp", "value").show(5)

## Identify Duplicates

Find records with same equipment, sensor, and timestamp

In [0]:
from pyspark.sql.functions import row_number, desc, col
from pyspark.sql.window import Window

# Define duplicate key: equipment_id + sensor_type + timestamp
# These three columns uniquely identify a single sensor reading

# Create windos to detect duplicates
window_spec = Window.partitionBy("equipment_id", "sensor_type", "timestamp").orderBy(desc("ingestion_timestamp")) #Keep latest ingestion

# Add row number - 1 = latest, 2+ duplicates
dedup_df = bronze_df.withColumn("rn", row_number().over(window_spec))

# Show duplicates
duplicates = dedup_df.filter(col("rn") > 1)
duplicate_count = duplicates.count()

if duplicate_count > 0:
    print(f"\nSample duplicates:")
    duplicates.select("equipment_id", "sensor_type", "timestamp", "value", "rn").show(10, truncate=False)



## Keep Only Latest Records

Keep row_number = 1 (latest) and drop duplicates

In [0]:
# Keep only the latest record (rn = 1)
deduplicated_df = dedup_df.filter(col("rn") == 1).drop("rn")

deduplicated_count = deduplicated_df.count()
removed_count = bronze_df.count() - deduplicated_df.count()

print(f"Records before dedup: {bronze_df.count()}")
print(f"Records after dedup: {deduplicated_count}")
print(f"Records removed: {removed_count}")

# Calculate dedup rate
dedup_rate = (removed_count / bronze_df.count()) * 100
print(f"\nDeduplication rate: {dedup_rate:.2f}%")

# Show sample of deduplicated data
print("\nSample after deduplication:")
deduplicated_df.select("equipment_id", "sensor_type", "timestamp", "value").show(5)

## Verify No Duplicates Remain

In [0]:
# Check for duplicates in output
remaining_duplicates = deduplicated_df.groupBy(
    "equipment_id", "sensor_type", "timestamp"
).count().filter("count > 1")

remaining_dup_count = remaining_duplicates.count()

print(f"Remaining duplicates: {remaining_dup_count}")

if remaining_dup_count == 0:
    print("All duplicates removed!")
else:
    print(f"Found {remaining_dup_count} remaining duplicates")
    remaining_duplicates.show(10)

## Add Deduplication Etadata
Track the deduplication process

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add metadata
final_df = deduplicated_df \
    .withColumn("silver_timestamp", current_timestamp()) \
    .withColumn("silver_version", lit("1.0")) \
    .withColumn("dedup_flag", lit("deduplicated"))

print("Silver metadata added")

## Write to Silver Table

In [0]:
print(f"Writing {final_df.count()} records to {OUTPUT_TABLE}")

final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE)

print(f"Silver table written")

## Verify Silver Table

In [0]:
# Read back silver table
silver_df = spark.read.table(OUTPUT_TABLE)

print(f"Silver table: {OUTPUT_TABLE}")
print(f"Record count: {silver_df.count()}")
print(f"\nSample records:")
silver_df.select("equipment_id", "sensor_type", "timestamp", "value").show(5)

# Show table info
spark.sql(f"DESCRIBE EXTENDED {OUTPUT_TABLE}") \
    .filter("col_name IN ('Catalog', 'Database', 'Table', 'Type', 'Provider')") \
    .show(truncate=False)

## Deduplication Summary Report

In [0]:
print("=" * 70)
print("DEDUPLICATION SUMMARY")
print("=" * 70)
print(f"Input table: {INPUT_TABLE}")
print(f"Output table: {OUTPUT_TABLE}")
print(f"")
print(f"METRICS")
print(f"Input records: {bronze_df.count():,}")
print(f"Output records: {deduplicated_count:,}")
print(f"Removed (dupes): {removed_count:,}")
print(f"Dedup rate: {dedup_rate:.2f}%")
print(f"")
print(f"QUALITY CHECK:")
print(f"Remaining dupes: {remaining_dup_count}")
print(f"Status: {'PASS' if remaining_dup_count == 0 else 'FAIL'}")
print(f"=" * 70)